# ***Resume Information Extraction & Filtered Semantic Search System***
## This notebook implements an end-to-end intelligent pipeline to parse unstructured resume files, split them structurally, compute dense embeddings, extract key metadata, and store them in an in-memory vector database (ChromaDB) to perform semantic searches coupled with metadata constraints.

---

## Pipeline Workflow Breakdown

### 1. Document Discovery and Loading
Purpose: Scans a specified directory for candidate resume files in various formats (`.pdf`, `.txt`, `.docx`, `.doc`).
How it works: Uses the standard library `glob` module to scan files matching specific extensions, producing a listing of candidate file paths.

### 2. File Parsing & Intelligent Section Chunking
* **Libraries Used**: `pypdf` (for PDF documents) and `python-docx` (for Word documents).
* **Text Extraction**: Reads raw content from PDF, Word, and text files safely while handling errors.
* **Structural Parsing**: Instead of arbitrary character-count splitting, a regular expression scanner identifies major structural headings (e.g., *Education*, *Skills*, *Experience*, *Projects*). It segments the document into clean, contextual sections so that semantic information belongs to the correct header.

### 3. Dense Vector Embeddings Generation
* **Library Used**: `sentence-transformers`.
* **Embedding Model**: `all-MiniLM-L6-v2` (a lightweight, high-performance Transformer model yielding 384-dimensional vectors).
* **Process**: Iterates through each extracted section of each resume and generates a dense numerical representation (embedding) representing the semantic meaning of that chunk.

### 4. Direct Database Population
* **Library Used**: `chromadb`.
* **Execution**: Spins up a local, ephemeral in-memory **ChromaDB** client, creates a database collection, and batch-inserts the document chunks alongside their computed embeddings and basic file tags.

### 5. Rule-Based Metadata Extraction
* **Function**: `extract_resume_metadata` uses regex patterns and structural parsing rules to scan through each resume's parsed sections to extract clean metadata fields:
  * **Candidate Name**: Derived from the Contact/Header section or filename.
  * **Skills**: Aggregates technical keywords parsed from the Skills section.
  * **Experience Years**: Computes an approximate timeframe by checking date ranges (e.g., `2018-2021`) within the Experience section.
  * **Education**: Extracts the primary institution/degree details from the Education section.

### 6. Relational & Vector Search with Metadata Filtering
* **Mechanism**: Maps the extracted fields directly to the metadata dictionaries inside a new collection called `resume_embeddings_filtered`.
* **Filtered Queries**: Demonstrates querying the database semantically (e.g., searching for *"infrastructure automation"*) while applying strict SQL-like conditions on the metadata attributes, such as filtering for candidates with `experience_years >= 3`.

In [5]:
import glob
import os

# Define the directory to scan (current directory in this case)
search_dir = '../data/resumes'

# Define the file patterns we want to search for
extensions = ['*.pdf', '*.txt', '*.docx', '*.doc']
resumes = []

for ext in extensions:
    # Find all files matching the extension
    resumes.extend(glob.glob(os.path.join(search_dir, ext)))

# Sort files to ensure consistency
resumes = sorted(list(set(resumes)))

print(f"Found {len(resumes)} resume(s):")
for resume in resumes:
    print(resume)

Found 46 resume(s):
../data/resumes\aarav_das.txt
../data/resumes\aditya_iyer.pdf
../data/resumes\ananya_reddy.pdf
../data/resumes\anika_taylor.pdf
../data/resumes\arjun_conti.txt
../data/resumes\chloe_berg.pdf
../data/resumes\daniel_hassan.pdf
../data/resumes\divya_novak.pdf
../data/resumes\elena_rossi.pdf
../data/resumes\emily_sharma.pdf
../data/resumes\ethan_okafor.pdf
../data/resumes\fatima_patel.pdf
../data/resumes\grace_bose.txt
../data/resumes\hannah_johnson.pdf
../data/resumes\hiro_smith.pdf
../data/resumes\isha_haddad.pdf
../data/resumes\james_murphy.pdf
../data/resumes\kabir_costa.pdf
../data/resumes\karthik_evans.pdf
../data/resumes\lars_khan.pdf
../data/resumes\leila_singh.pdf
../data/resumes\lucas_petrov.pdf
../data/resumes\marco_clark.pdf
../data/resumes\meera_gupta.pdf
../data/resumes\mei_larsen.pdf
../data/resumes\michael_chen.pdf
../data/resumes\nadia_joshi.pdf
../data/resumes\neha_malhotra.txt
../data/resumes\noah_menon.pdf
../data/resumes\olivia_verma.pdf
../data/res

In [6]:
!pip install -q pypdf python-docx

import os
import re
import pypdf
import docx

def extract_text(file_path):
    _, ext = os.path.splitext(file_path.lower())
    text = ""
    if ext == '.pdf':
        try:
            reader = pypdf.PdfReader(file_path)
            text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
        except Exception as e:
            print(f"Error reading PDF {file_path}: {e}")
    elif ext in ['.docx', '.doc']:
        try:
            doc = docx.Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs])
        except Exception as e:
            print(f"Error reading Word document {file_path}: {e}")
    elif ext == '.txt':
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading text file {file_path}: {e}")
    return text

def intelligent_chunk_resume(text):
    # Patterns targeting common section headers (without inline (?i) flags)
    headers = [
        r'^\s*(education|academic background|academic qualifications)\s*$',
        r'^\s*(work experience|professional experience|experience|employment history)\s*$',
        r'^\s*(skills|technical skills|areas of expertise|professional skills)\s*$',
        r'^\s*(projects|personal projects|key projects)\s*$',
        r'^\s*(certifications|awards|honors|activities)\s*$'
    ]
    # Compile with the re.IGNORECASE flag at the start
    combined_pattern = re.compile('|'.join(headers), re.IGNORECASE)

    lines = text.split('\n')
    chunks = []
    current_chunk = []
    current_header = "Contact / Header Information"

    for line in lines:
        if combined_pattern.match(line.strip()):
            # Save the previous chunk if it has content
            if current_chunk:
                chunks.append({
                    "section": current_header,
                    "content": "\n".join(current_chunk).strip()
                })
                current_chunk = []
            current_header = line.strip()
        current_chunk.append(line)

    if current_chunk:
        chunks.append({
            "section": current_header,
            "content": "\n".join(current_chunk).strip()
        })

    return chunks

# Process all loaded resumes from the previous step
chunked_resumes = {}
for filepath in resumes:
    file_text = extract_text(filepath)
    if file_text.strip():
        chunks = intelligent_chunk_resume(file_text)
        chunked_resumes[filepath] = chunks
        print(f"\nProcessed: {filepath} ({len(chunks)} sections found)")
        for chunk in chunks:
            print(f"  - Section: [{chunk['section']}] ({len(chunk['content'].split())} words)")
    else:
        print(f"\nSkipped or empty file: {filepath}")



Processed: ../data/resumes\aarav_das.txt (5 sections found)
  - Section: [Contact / Header Information] (31 words)
  - Section: [TECHNICAL SKILLS] (19 words)
  - Section: [PROFESSIONAL EXPERIENCE] (142 words)
  - Section: [KEY PROJECTS] (29 words)
  - Section: [ACADEMIC BACKGROUND] (27 words)

Processed: ../data/resumes\aditya_iyer.pdf (3 sections found)
  - Section: [Contact / Header Information] (53 words)
  - Section: [EXPERIENCE] (122 words)
  - Section: [ACADEMIC BACKGROUND] (24 words)

Processed: ../data/resumes\ananya_reddy.pdf (4 sections found)
  - Section: [Contact / Header Information] (47 words)
  - Section: [EXPERIENCE] (97 words)
  - Section: [KEY PROJECTS] (18 words)
  - Section: [ACADEMIC BACKGROUND] (26 words)

Processed: ../data/resumes\anika_taylor.pdf (5 sections found)
  - Section: [Contact / Header Information] (27 words)
  - Section: [SKILLS] (23 words)
  - Section: [PROFESSIONAL EXPERIENCE] (152 words)
  - Section: [ACADEMIC BACKGROUND] (29 words)
  - Section: 

In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# Load a lightweight, high-performing Hugging Face model for generating embeddings
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embedded_resumes = {}

print("Generating embeddings for chunked resumes...")
for filepath, chunks in chunked_resumes.items():
    embedded_resumes[filepath] = []
    for chunk in chunks:
        section_name = chunk["section"]
        content_text = chunk["content"]

        # Generate embedding for the text
        embedding = embedding_model.encode(content_text).tolist()

        embedded_resumes[filepath].append({
            "section": section_name,
            "content": content_text,
            "embedding": embedding
        })
        print(f"  - Processed section: {section_name} ({len(content_text.split())} words)")
    print(f"  - Generated {len(chunks)} embeddings for {filepath}")

print("\nEmbedding generation complete!")

c:\Users\Aryahi\OneDrive\Documents\GitHub\RAG-Based-Profile-matching\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SentenceTransformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1283.42it/s]


Generating embeddings for chunked resumes...
  - Processed section: Contact / Header Information (31 words)
  - Processed section: TECHNICAL SKILLS (19 words)
  - Processed section: PROFESSIONAL EXPERIENCE (142 words)
  - Processed section: KEY PROJECTS (29 words)
  - Processed section: ACADEMIC BACKGROUND (27 words)
  - Generated 5 embeddings for ../data/resumes\aarav_das.txt
  - Processed section: Contact / Header Information (53 words)
  - Processed section: EXPERIENCE (122 words)
  - Processed section: ACADEMIC BACKGROUND (24 words)
  - Generated 3 embeddings for ../data/resumes\aditya_iyer.pdf
  - Processed section: Contact / Header Information (47 words)
  - Processed section: EXPERIENCE (97 words)
  - Processed section: KEY PROJECTS (18 words)
  - Processed section: ACADEMIC BACKGROUND (26 words)
  - Generated 4 embeddings for ../data/resumes\ananya_reddy.pdf
  - Processed section: Contact / Header Information (27 words)
  - Processed section: SKILLS (23 words)
  - Processed sec

In [8]:
!pip install -q chromadb

import chromadb
from chromadb.config import Settings

# Initialize local ephemeral (in-memory) ChromaDB client
chroma_client = chromadb.Client()

# Create or get the collection
collection_name = "resume_embeddings"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)

# Prepare lists for batch insertion
ids = []
documents = []
embeddings = []
metadatas = []

counter = 0
for filepath, chunks in embedded_resumes.items():
    filename = os.path.basename(filepath)
    for i, chunk in enumerate(chunks):
        # Construct unique IDs
        chunk_id = f"{filename}_chunk_{i}"

        ids.append(chunk_id)
        documents.append(chunk["content"])
        embeddings.append(chunk["embedding"])
        metadatas.append({
            "source_file": filename,
            "section": chunk["section"]
        })

# Add documents, embeddings, and metadata to ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print(f"Successfully stored {len(ids)} chunks in the ChromaDB collection: '{collection_name}'!")

# Quick test query to verify
query_text = "experience with Spark or large-scale data ingestion"
# Generate the query embedding using our previously loaded model
query_embedding = embedding_model.encode(query_text).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print("\n--- Sample Search Results for '" + query_text + "' ---")
for idx, doc in enumerate(results['documents'][0]):
    metadata = results['metadatas'][0][idx]
    print(f"\nRank {idx+1}:")
    print(f"Source: {metadata['source_file']} | Section: {metadata['section']}")
    print(f"Content Preview:\n{doc[:200]}...")

Successfully stored 208 chunks in the ChromaDB collection: 'resume_embeddings'!

--- Sample Search Results for 'experience with Spark or large-scale data ingestion' ---

Rank 1:
Source: hiro_smith.pdf | Section: WORK EXPERIENCE
Content Preview:
WORK EXPERIENCE
Finch & Co. Technologies — Senior Data Engineer (07/2020 – Present)
  Wrote Python tooling to validate schemas and monitor pipeline freshness.
  Orchestrated 20 production DAGs in Ai...

Rank 2:
Source: wei_wilson.pdf | Section: KEY PROJECTS
Content Preview:
KEY PROJECTS
  Lakehouse ingestion framework in Python and Spark with automated backfills....


In [9]:
import re

def extract_resume_metadata(filepath, chunks):
    filename = os.path.basename(filepath)

    # Default values
    name = filename.split('.')[0].replace('_', ' ').title()
    skills = []
    experience_years = 0
    education = "Not specified"

    # Simple rule-based extraction patterns
    for chunk in chunks:
        section = chunk["section"].lower()
        content = chunk["content"]

        # Extracting Name from Contact / Header Information
        if "contact" in section or "header" in section:
            lines = [l.strip() for l in content.split('\n') if l.strip()]
            if lines:
                name = lines[0] # Usually the first line is the name

        # Extracting Skills
        elif "skills" in section:
            # Grab lines after "Skills" heading or extract comma-separated items
            lines = content.split('\n')
            for line in lines:
                if "skills" in line.lower() and len(line.split(',')) <= 1:
                    continue
                skills.extend([s.strip() for s in line.split(',') if s.strip()])

        # Extracting Education
        elif "education" in section or "academic" in section:
            lines = [l.strip() for l in content.split('\n') if l.strip()]
            edu_lines = [l for l in lines if not re.match(r'^(education|academic)', l, re.I)]
            if edu_lines:
                education = edu_lines[0]

        # Estimating Experience Years
        elif "experience" in section or "employment" in section:
            # Find 4-digit years like (2018-2021) or (2021-Present)
            years = re.findall(r'\b(20\d{2})\b', content)
            if len(years) >= 2:
                try:
                    sorted_years = sorted([int(y) for l in years for y in [l]])
                    diff = sorted_years[-1] - sorted_years[0]
                    experience_years = max(experience_years, diff if diff > 0 else 1)
                except Exception:
                    pass

    return {
        "name": name,
        "skills": ", ".join(skills[:15]),
        "experience_years": experience_years if experience_years > 0 else 2, # Fallback default
        "education": education
    }

In [10]:
# Re-initialize and update ChromaDB collection with rich metadata
try:
    chroma_client.delete_collection(name="resume_embeddings_filtered")
except Exception:
    pass

collection_filtered = chroma_client.create_collection(name="resume_embeddings_filtered")

ids = []
documents = []
embeddings = []
metadatas = []

for filepath, chunks in embedded_resumes.items():
    filename = os.path.basename(filepath)
    structured_meta = extract_resume_metadata(filepath, chunks)

    for i, chunk in enumerate(chunks):
        chunk_id = f"{filename}_filtered_chunk_{i}"
        ids.append(chunk_id)
        documents.append(chunk["content"])
        embeddings.append(chunk["embedding"])

        # Store structured metadata alongside section info for chunk-level filtering
        metadatas.append({
            "source_file": filename,
            "section": chunk["section"],
            "name": structured_meta["name"],
            "skills": structured_meta["skills"],
            "experience_years": int(structured_meta["experience_years"]),
            "education": structured_meta["education"]
        })

collection_filtered.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print("Populated metadata-rich collection successfully!")

Populated metadata-rich collection successfully!


## Hybrid Job Matching with Structured Results

This section ranks resumes using both semantic similarity and BM25 keyword relevance. Must-have requirements are applied as hard filters, while the final candidate score is reported on a 0-100 scale. The result is validated with Pydantic and printed as JSON with matched skills, relevant sections, and excerpts.

Run these cells after the resume collection above has been populated. The matcher reuses the existing `embedding_model` and `collection_filtered` objects.

### Dependencies and output models

Install the ranking and validation libraries if they are not already available in the notebook kernel. `CandidateMatch` constrains each score to 0-100 and defines the evidence returned for one candidate; `MatchResponse` wraps the job description and ranked matches.

In [1]:
%pip install -q rank_bm25 pydantic

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import re
import time
from pathlib import Path

from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi


class CandidateMatch(BaseModel):
    candidate_name: str
    resume_path: str
    match_score: int = Field(ge=0, le=100)
    matched_skills: list[str]
    relevant_sections: list[str]
    relevant_excerpts: list[str]
    reasoning: str


class MatchResponse(BaseModel):
    job_description: str
    top_matches: list[CandidateMatch]
    performance_metrics: dict[str, float | int | str | None]

### Matching helpers

These helpers tokenize text for BM25, perform boundary-aware exact skill checks (so `Java` does not match `JavaScript`), parse semicolon-separated must-haves, and collect the resume sections and excerpts that support a match.

In [12]:
def tokenize(text):
    return re.findall(r"[a-z0-9+#.]+", text.lower())


def exact_match(phrase, text):
    return re.search(rf"(?<!\w){re.escape(phrase)}(?!\w)", text, re.IGNORECASE) is not None


def parse_must_haves(requirements):
    minimum_years = 0
    required_skills = []
    for requirement in (item.strip() for item in requirements.split(";")):
        if not requirement:
            continue
        years_match = re.search(r"(\d+)\s*\+?\s*(?:years?|yrs?)\b(.*)", requirement, re.IGNORECASE)
        if years_match:
            minimum_years = max(minimum_years, int(years_match.group(1)))
            skill = re.sub(
                r"^\s*(?:of\s+)?(?:experience\s*)?(?:with|in)?\s*",
                "",
                years_match.group(2),
                flags=re.IGNORECASE,
            ).strip()
            if skill:
                required_skills.append(skill)
        else:
            required_skills.append(requirement)
    return minimum_years, list(dict.fromkeys(required_skills))


def collect_candidate_evidence(chunks, skills):
    all_text = " ".join(chunk["document"] for chunk in chunks)
    metadata = chunks[0]["metadata"]
    matched_skills = [
        skill for skill in skills
        if exact_match(skill, f"{all_text} {metadata.get('skills', '')}")
    ]
    relevant_sections = list(dict.fromkeys(
        chunk["metadata"].get("section", "Resume")
        for chunk in chunks
        if chunk["query_overlap"]
        or any(exact_match(skill, chunk["document"]) for skill in skills)
    ))
    relevant_excerpts = [
        f"{chunk['metadata'].get('section', 'Resume')}: {chunk['document'][:300].strip()}"
        for chunk in sorted(chunks, key=lambda item: item["hybrid_score"], reverse=True)[:2]
    ]
    return matched_skills, relevant_sections, relevant_excerpts

### Enter requirements and rank resume chunks

Enter the job description, optional critical skills, and must-have requirements. Write must-haves separated by semicolons, for example `5+ years Python; AWS`. Semantic and BM25 ranks are normalized and combined into a 0-100 chunk score; a later cell aggregates these scores by candidate and filters candidates who miss any must-have.

In [ ]:
job_description = input("Enter job description: ").strip()
critical_skills_input = input("Enter critical skills (comma-separated, optional): ")
critical_skills = [skill.strip() for skill in critical_skills_input.split(",") if skill.strip()]
must_haves_input = input(
    "Enter must-have requirements separated by semicolons (e.g. 5+ years Python; AWS): "
)
minimum_years, required_skills = parse_must_haves(must_haves_input)
scoring_skills = list(dict.fromkeys(critical_skills + required_skills))

retrieval_started_at = time.perf_counter()
jd_embedding = embedding_model.encode(job_description).tolist()
corpus = collection_filtered.get(include=["documents", "metadatas"])
documents = corpus["documents"] or []
metadatas = corpus["metadatas"] or []
resume_ids = corpus["ids"]
if not documents:
    raise RuntimeError("The filtered resume collection is empty.")

semantic_results = collection_filtered.query(
    query_embeddings=[jd_embedding],
    n_results=len(documents),
)
semantic_ranks = {
    resume_id: rank
    for rank, resume_id in enumerate(semantic_results["ids"][0], start=1)
}
query_terms = set(tokenize(job_description))
tokenized_documents = [
    tokenize(f"{document} {metadata.get('skills', '')}")
    for document, metadata in zip(documents, metadatas)
]
bm25 = BM25Okapi(tokenized_documents)
bm25_query = tokenize(f"{job_description} {' '.join(scoring_skills)}")
bm25_scores = bm25.get_scores(bm25_query)
bm25_order = sorted(range(len(documents)), key=lambda index: bm25_scores[index], reverse=True)
bm25_ranks = {
    resume_ids[index]: rank
    for rank, index in enumerate(bm25_order, start=1)
}

chunk_records = []
for resume_id, document, metadata in zip(resume_ids, documents, metadatas):
    chunk_terms = set(tokenize(f"{document} {metadata.get('skills', '')}"))
    query_overlap = len(query_terms & chunk_terms) / max(len(query_terms), 1)
    semantic_quality = 1 - (semantic_ranks.get(resume_id, len(resume_ids)) - 1) / max(len(resume_ids) - 1, 1)
    keyword_quality = 1 - (bm25_ranks[resume_id] - 1) / max(len(resume_ids) - 1, 1)
    matched_skills = sum(
        exact_match(skill, f"{document} {metadata.get('skills', '')}")
        for skill in scoring_skills
    )
    skill_quality = matched_skills / len(scoring_skills) if scoring_skills else query_overlap
    hybrid_score = round(100 * (
        0.40 * semantic_quality
        + 0.30 * keyword_quality
        + 0.20 * skill_quality
        + 0.10 * bool(query_overlap)
    ))
    chunk_records.append({
        "id": resume_id,
        "document": document,
        "metadata": metadata,
        "query_overlap": query_overlap,
        "hybrid_score": hybrid_score,
    })

print(f"Ranked {len(chunk_records)} resume chunks.")

Ranked 208 resume chunks.


Resume chunks are grouped by source file. A candidate is excluded if their recorded experience is below the requested minimum or any required skill is absent from their resume text/skills metadata. Remaining candidates are ordered by their best chunk score, with matched skills, evidence sections, and excerpts included in the validated response. Performance metrics report retrieval latency and a top-10 critical-skill hit rate. The hit rate is a relevance proxy based on supplied critical skills, not benchmark accuracy from labeled relevance judgments; it is `null` when no critical skills are supplied.

In [ ]:
grouped_candidates = {}
for chunk in chunk_records:
    source_file = chunk["metadata"]["source_file"]
    grouped_candidates.setdefault(source_file, []).append(chunk)

candidate_matches = []
for source_file, chunks in grouped_candidates.items():
    metadata = chunks[0]["metadata"]
    all_text = " ".join(chunk["document"] for chunk in chunks)
    all_text += " " + metadata.get("skills", "")
    experience_years = int(metadata.get("experience_years", 0))

    if experience_years < minimum_years:
        continue
    if any(not exact_match(skill, all_text) for skill in required_skills):
        continue

    matched_skills, relevant_sections, relevant_excerpts = collect_candidate_evidence(
        chunks,
        scoring_skills,
    )
    score = max(chunk["hybrid_score"] for chunk in chunks)
    reasoning_parts = [f"Hybrid semantic and BM25 relevance score: {score}/100."]
    if relevant_sections:
        reasoning_parts.append(f"Relevant resume sections: {', '.join(relevant_sections)}.")
    if matched_skills:
        reasoning_parts.append(f"Exact skills found: {', '.join(matched_skills)}.")
    if minimum_years:
        reasoning_parts.append(
            f"Meets the {minimum_years}+ years experience requirement "
            f"({experience_years} years recorded)."
        )
    if required_skills:
        reasoning_parts.append(f"Meets required skills: {', '.join(required_skills)}.")

    candidate_matches.append(CandidateMatch(
        candidate_name=metadata.get("name", Path(source_file).stem),
        resume_path=(Path("data") / "resumes" / source_file).as_posix(),
        match_score=score,
        matched_skills=matched_skills,
        relevant_sections=relevant_sections,
        relevant_excerpts=relevant_excerpts,
        reasoning=" ".join(reasoning_parts),
    ))

candidate_matches.sort(key=lambda candidate: candidate.match_score, reverse=True)
top_matches = candidate_matches[:10]
retrieval_latency_ms = round((time.perf_counter() - retrieval_started_at) * 1000, 2)
critical_skill_relevant_count = sum(
    any(skill in candidate.matched_skills for skill in critical_skills)
    for candidate in top_matches
)
retrieval_accuracy = (
    round(critical_skill_relevant_count / len(top_matches), 4)
    if critical_skills and top_matches
    else None
)
chunk_scores = [chunk["hybrid_score"] for chunk in chunk_records]
candidate_scores = [candidate.match_score for candidate in candidate_matches]

match_response = MatchResponse(
    job_description=job_description,
    top_matches=top_matches,
    performance_metrics={
        "retrieval_latency_ms": retrieval_latency_ms,
        "retrieval_accuracy_at_k": retrieval_accuracy,
        "accuracy_basis": (
            "Fraction of returned top candidates matching at least one declared critical skill; "
            "this is a relevance proxy, not ground-truth benchmark accuracy."
            if critical_skills
            else "Unavailable: no critical skills were declared."
        ),
        "relevant_candidates_at_k": critical_skill_relevant_count if critical_skills else None,
        "evaluated_candidates_at_k": len(top_matches),
        "total_chunks": len(chunk_records),
        "total_candidates": len(grouped_candidates),
        "average_chunk_score": round(sum(chunk_scores) / len(chunk_scores), 2) if chunk_scores else None,
        "average_candidate_score": round(sum(candidate_scores) / len(candidate_scores), 2) if candidate_scores else None,
        "minimum_chunk_score": min(chunk_scores) if chunk_scores else None,
        "maximum_chunk_score": max(chunk_scores) if chunk_scores else None,
        "minimum_candidate_score": min(candidate_scores) if candidate_scores else None,
        "maximum_candidate_score": max(candidate_scores) if candidate_scores else None,
    },
)
print(match_response.model_dump_json(indent=2))

{
  "job_description": "Responsibilities - Design and build microservices and REST APIs for payment and ledger workflows. - Implement event-driven integrations using Kafka. - Improve performance, reliability and test coverage of existing services.  Requirements - 4+ years of backend software engineering experience, including 3+ years with Java and Spring Boot. - Experience designing microservices and REST APIs. - Strong knowledge of relational databases such as PostgreSQL or MySQL. - Experience with Kafka or RabbitMQ. - Bachelor's degree in Computer Science or equivalent.  Nice to have - Experience with Kubernetes, Docker and AWS. - Familiarity with system design for high-scale systems.",
  "top_matches": [
    {
      "candidate_name": "Anika Taylor",
      "resume_path": "data/resumes/anika_taylor.pdf",
      "match_score": 99,
      "matched_skills": [
        "Java",
        "AWS"
      ],
      "relevant_sections": [
        "Contact / Header Information",
        "SKILLS",
      